# Django Forms & Views Laboratory

> Мета — простежити шлях **однієї нотатки**: від браузера через Form, View і Service до бази даних.

---

## Головна карта

```
Browser
  ↓ (POST: title=..., priority=...)
request.POST          ← рядки! все небезпечно
  ↓
Form (is_valid())     ← перевірка і конвертація типів
  ↓
cleaned_data          ← перевірені Python-об'єкти
  ↓
View.form_valid()     ← приймає рішення
  ↓
services.create_note()← зберігає в базу
  ↓
Database
  ↓
redirect              ← користувач переходить на нову сторінку
```

Ця карта — твій GPS на весь notebook. Повертайся до неї щоразу коли губишся.

## Що ми будуємо?

Уяви: ти розробник Notes App. Користувач хоче:

1. Відкрити сторінку `/notes/new/`
2. Ввести назву нотатки
3. Вибрати записник і теги
4. Натиснути кнопку "Зберегти"
5. Побачити свою нотатку — або помилки, якщо щось не так

**Питання:** що відбувається між кроком 4 і кроком 5?

Саме це ми і розбираємо у цій лабораторії.

Django для цього використовує:

    Form → View → Template → Service → Database

## Три рівні лабораторії

| Рівень | Частини | Теми |
|--------|---------|------|
| 🟢 **Beginner** | 0 – 3 | Ментальна модель, SimpleForm, Trust Boundary, NotebookForm |
| 🟡 **Intermediate** | 4 – 7 | TagForm.clean\_name(), NoteForm + FK/M2M, Widgets, CBV GET/POST |
| 🔴 **Advanced** | 8 – 10 | Test Client, CSRF, Testing Matrix |

Перший прохід: пройди рівні послідовно.  
Другий прохід: кожна частина незалежна — використовуй як довідник.

---
## 0. Setup — Django Initialization

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'hello_project.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

print('✅ Django готовий!')
print(f'   DB: {django.conf.settings.DATABASES["default"]["NAME"]}')
print(f'   Apps: {len(django.apps.registry.apps.get_app_configs())} зареєстровано')

In [ ]:
# Імпорти — все що потрібно для лабораторії
from django.test import RequestFactory, Client
from django.contrib.auth.models import User
from django.db import transaction
from django.urls import reverse

from hello_app.forms import NoteForm, NotebookForm, TagForm
from hello_app.models import Note, Notebook, Tag
from hello_app import views, services, selectors


# ── Хелпер: відображення стану форми ─────────────────────────────────────────
def show_form(form, label="Form"):
    """Показує стан форми: bound/unbound, errors, cleaned_data."""
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Bound (є дані?):  {form.is_bound}")
    if form.is_bound:
        valid = form.is_valid()
        print(f"  is_valid():       {valid}")
        if form.errors:
            print(f"  Errors:           {dict(form.errors)}")
        if hasattr(form, 'cleaned_data') and form.cleaned_data:
            print(f"  cleaned_data:")
            for k, v in form.cleaned_data.items():
                print(f"    {k:15} = {v!r}  ({type(v).__name__})")
    print()


# ── Хелпер: відображення response ────────────────────────────────────────────
def show_response(response, label="Response"):
    """Показує HTTP response: status, redirect, context keys."""
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Status code:  {response.status_code}")
    if hasattr(response, 'url'):
        print(f"  Redirect URL: {response.url}")
    if hasattr(response, 'context') and response.context:
        ctx = response.context
        if hasattr(ctx, 'keys'):
            print(f"  Context keys: {list(ctx.keys())}")
    if hasattr(response, 'templates') and response.templates:
        print(f"  Templates:    {[t.name for t in response.templates]}")
    print()


print('✅ Імпорти та хелпери готові')

In [ ]:
# ── Seed data: створюємо тестові дані ────────────────────────────────────────
with transaction.atomic():
    User.objects.filter(username='lab_user').delete()

    user = User.objects.create_user(
        username='lab_user',
        password='testpass123',
        email='lab@test.com'
    )

    notebook = services.create_notebook(
        user=user,
        title='Lab Notebook',
        color='#4A90E2'
    )

    tag_python, _ = services.create_or_get_tag(user=user, name='python', color='#FF5733')
    tag_django, _ = services.create_or_get_tag(user=user, name='django', color='#2ECC71')

    note = services.create_note(
        user=user,
        title='Перша нотатка',
        content='Вміст для тестування',
        priority=2,
        notebook=notebook,
        tag_ids=[tag_python.pk]
    )

print('Seed data готові:')
print(f'   user:     {user.username} (id={user.pk})')
print(f'   notebook: {notebook.title} (id={notebook.pk})')
print(f'   tags:     {tag_python.name}, {tag_django.name}')
print(f'   note:     {note.title} (id={note.pk})')

---
## Частина 1 — Що таке форма?
### Рівень: 🟢 Beginner

Форма Django — це **перевіряльник даних**.

Коли користувач натискає "Зберегти", браузер надсилає:

    POST /notes/new/
    title=&content=&priority=99

Це просто рядки. Django не знає чи `priority=99` — правильне значення.

**Форма відповідає на три питання:**
1. Чи всі обов'язкові поля заповнені?
2. Чи правильний тип даних (рядок, число, email)?
3. Чи виконуються бізнес-правила (значення у допустимому діапазоні)?

Перш ніж знайомитись з `ModelForm`, подивимось на найпростішу форму — `forms.Form`.

### Навіщо це потрібно?

Якщо передавати `request.POST` напряму в ORM — будь-хто може надіслати будь-що:  
`priority=999`, `user_id=2`, `is_admin=true` — і Django збереже це без питань.

Форма — перша лінія захисту.

In [ ]:
# forms.Form — форма без прив'язки до моделі
# Поля описуються вручну — ніяких моделей, ніякої магії
from django import forms as django_forms

class SimpleNoteForm(django_forms.Form):
    title    = django_forms.CharField(max_length=200, label='Заголовок')
    priority = django_forms.IntegerField(min_value=1, max_value=4, label='Пріоритет')

print('✅ SimpleNoteForm визначено')
print(f'Поля: {list(SimpleNoteForm().fields.keys())}')
print()
print('Кожне поле — це об\'єкт:')
for name, field in SimpleNoteForm().fields.items():
    print(f'  {name:10} → {type(field).__name__}')

In [ ]:
# Unbound форма — немає даних (GET запит)
# Так виглядає форма коли користувач щойно відкрив сторінку /notes/new/

form_empty = SimpleNoteForm()

print('=== Unbound форма (GET запит) ===')
print(f'is_bound:     {form_empty.is_bound}')    # False — немає POST даних
print(f'is_valid():   {form_empty.is_valid()}')   # False — немає чого перевіряти
print(f'errors:       {dict(form_empty.errors)}') # {} — нема помилок (і немає даних)

### Checkpoint 1

**Питання:** Чому `is_valid()` повертає `False` для unbound форми, навіть якщо помилок немає?

<details>
<summary>▶ Відповідь</summary>

`is_valid()` перевіряє два критерії: `is_bound AND not self.errors`.  
Якщо `is_bound = False` → форма автоматично невалідна, бо перевіряти нічого.  
Щоб форма стала **bound** — треба передати словник даних як перший аргумент:

    form = SimpleNoteForm({"title": "Тест", "priority": "2"})

</details>

In [ ]:
# Bound форма з валідними даними — симулюємо POST запит
# Зверни увагу: priority передається як РЯДОК '2', не число 2
# (браузер завжди надсилає рядки!)

form_valid = SimpleNoteForm({'title': 'Моя нотатка', 'priority': '2'})

print('=== Bound форма з валідними даними ===')
print(f'is_bound:     {form_valid.is_bound}')    # True
print(f'is_valid():   {form_valid.is_valid()}')   # True
print()
print('cleaned_data:')
for key, value in form_valid.cleaned_data.items():
    print(f'  {key:12} = {value!r}  (тип: {type(value).__name__})')
print()
print("Зверни увагу: priority = 2 (int), а не '2' (str)!")
print('Форма перетворила рядок у ціле число.')

In [ ]:
# Невалідні дані: порожній title (required) + priority поза діапазоном

form_invalid = SimpleNoteForm({'title': '', 'priority': '99'})

print('=== Bound форма з невалідними даними ===')
print(f'is_bound:    {form_invalid.is_bound}')
print(f'is_valid():  {form_invalid.is_valid()}')
print()
print('errors:')
for field, errors in form_invalid.errors.items():
    print(f'  {field}: {list(errors)}')
print()
if hasattr(form_invalid, 'cleaned_data'):
    print('cleaned_data (лише валідні поля):', form_invalid.cleaned_data)
else:
    print('cleaned_data: відсутній (форма невалідна)')

### Висновок: Частина 1

```
request.POST  →  form.is_valid()  →  cleaned_data
  (рядки)            ↑                 (Python)
                перевіряє і
                конвертує типи
```

| Концепція | Що це означає |
|-----------|---------------|
| `is_bound` | Чи є дані для перевірки? |
| `is_valid()` | Чи всі дані пройшли перевірку? |
| `errors` | Словник помилок по полях |
| `cleaned_data` | Перевірені дані у правильних Python-типах |

**Далі:** `ModelForm` робить те саме, але поля беруться автоматично з моделі.

---

## Частина 2 — Trust Boundary: request.POST vs cleaned_data
### Рівень: 🟢 Beginner

**Форма Django = Кордон довіри (Trust Boundary)**

Уяви митницю між небезпечною і безпечною зонами:

```
🔴 НЕБЕЗПЕЧНА ЗОНА              🟢 БЕЗПЕЧНА ЗОНА
request.POST (рядки!)   →      form.cleaned_data (Python-об'єкти)

request.POST['priority'] = '2'  → cleaned_data['priority'] = 2     (int!)
request.POST['notebook'] = '5'  → cleaned_data['notebook'] = <Notebook>  (instance!)
request.POST['title'] = ''      → ValidationError (required!)
request.POST['priority'] = 'abc'→ ValidationError (not a valid choice!)
```

Ніколи не передавай `request.POST['field']` напряму в ORM — тільки через `form.cleaned_data`!

### Що ми зараз перевіряємо?

Як `NoteForm` (ModelForm) перетворює рядки з браузера у Python-об'єкти.

In [ ]:
# ── Bound form з валідними даними ──────────────────────────────────────────────
# Симулюємо POST запит: передаємо словник як перший аргумент

valid_data = {
    'title': 'Нова нотатка з форми',
    'content': 'Зміст нотатки',
    'priority': '2',              # ← рядок! з браузера завжди рядок
    'notebook': str(notebook.pk), # ← рядок! pk записника
    'tags': [str(tag_python.pk)], # ← список рядків!
    'is_pinned': False,
}

bound_valid = NoteForm(valid_data, user=user)

show_form(bound_valid, "Bound NoteForm — валідні дані")

print('Зверни увагу на типи у cleaned_data:')
print("  priority: '2' (рядок) → 2 (int)")
print("  notebook: 'pk' (рядок) → <Notebook> (instance!)")
print("  tags: ['pk'] (список рядків) → QuerySet (queryset!)")

In [ ]:
# ── Bound form з невалідними даними ────────────────────────────────────────────
# Порожній title → required field error

invalid_data = {
    'title': '',          # ← порожній! required поле
    'content': 'Зміст',
    'priority': '2',
    'notebook': '',
}

bound_invalid = NoteForm(invalid_data, user=user)

show_form(bound_invalid, "Bound NoteForm — порожній title")

print(f'Помилки для title: {list(bound_invalid.errors["title"])}')
print()
print('cleaned_data НЕ містить поле title — воно не пройшло валідацію!')

In [ ]:
# ── Невалідний вибір priority ────────────────────────────────────────────────────
# choices дозволяють тільки 1, 2, 3, 4 — передаємо '99'

bad_priority = {
    'title': 'Тест',
    'content': '',
    'priority': '99',  # ← не входить у choices!
    'notebook': '',
}

form_bad = NoteForm(bad_priority, user=user)
form_bad.is_valid()

print(f'Помилки: {dict(form_bad.errors)}')
print()
print(f'priority field type: {type(form_bad.fields["priority"]).__name__}')
print('TypedChoiceField перевіряє що значення є у choices → ValidationError якщо ні')

### Checkpoint 2

**Питання:** Яке значення `cleaned_data['priority']` якщо користувач надіслав `'1'`? А якщо `'99'`?

<details>
<summary>▶ Відповідь</summary>

- `'1'` → `cleaned_data['priority'] = 1` (int) — валідне значення з choices
- `'99'` → `cleaned_data` **не містить** поле `priority` — TypedChoiceField кидає ValidationError

Правило: якщо поле не пройшло валідацію — його **немає** у `cleaned_data`.

</details>

---

## Частина 3 — NotebookForm: найпростіший ModelForm
### Рівень: 🟢 Beginner

### Чому NotebookForm — ідеальний перший ModelForm?

`ModelForm` бере поля **автоматично з моделі** — не потрібно описувати вручну як у `forms.Form`.

Порівняння:

| | `forms.Form` | `forms.ModelForm` |
|-|-|-|
| Поля | Описуються вручну | Генеруються з моделі |
| `save()` | Немає | Є (але ми не використовуємо — є services) |
| Підходить для | Пошук, фільтри | Створення/редагування об'єктів |

`NotebookForm` — найпростіший ModelForm для початку, бо:
- Тільки прості поля: `title`, `description`, `color`, `is_default`
- **Без** ForeignKey і ManyToMany
- **Без** фільтрації по user у `__init__`

### Схема полів NotebookForm

```
Notebook модель          NotebookForm           HTML widget
────────────────         ────────────           ───────────
title (CharField)     →  CharField          →   <input type="text">
description (Text)    →  CharField          →   <textarea>
color (CharField)     →  CharField          →   <input type="color">  ← HTML5!
is_default (Boolean)  →  BooleanField       →   <input type="checkbox">
```

In [ ]:
# Досліджуємо NotebookForm — поля і їх типи

notebook_form_empty = NotebookForm()

print('NotebookForm.fields:')
for name, field in notebook_form_empty.fields.items():
    widget_name = type(field.widget).__name__
    print(f'  {name:15} → {type(field).__name__:20} (widget: {widget_name})')
print()
print('Метадані полів:')
for name, field in notebook_form_empty.fields.items():
    label = field.label
    required = field.required
    print(f'  {name:15} label={label!r:25} required={required}')

In [ ]:
# HTML5 color picker — поле color рендерить <input type="color">
# Це звичайний TextInput але з атрибутом type="color"

color_html = str(notebook_form_empty['color'])
print('form["color"] HTML:')
print(color_html)
print()
print('Ключовий атрибут: type="color"')
print('При кліку браузер показує палітру кольорів і повертає HEX рядок: #RRGGBB')

In [ ]:
# Валідний POST → cleaned_data
# Зверни увагу на типи: color і title — str, is_default — bool

notebook_post = {
    'title': 'Мій перший записник',
    'description': 'Тестовий опис',
    'color': '#FF5733',
    'is_default': 'on',  # ← чекбокс: браузер надсилає 'on' або нічого
}

notebook_form_valid = NotebookForm(notebook_post)

show_form(notebook_form_valid, "NotebookForm — валідні дані")
print('Зверни увагу: is_default = True (bool), а не "on" (str)!')

### Checkpoint 3

**Питання:** Які поля **не** включені у `NotebookForm` і чому?

<details>
<summary>▶ Відповідь</summary>

Не включені: `user`, `created_at`.

- `user` — встановлюється сервером у `services.create_notebook(user=request.user)`.  
  Якби `user` був у формі, Alice могла б надіслати `user_id=2` і створити записник від імені Bob!
- `created_at` — генерується автоматично Django (auto_now_add=True).

Правило: поля що встановлює **сервер** — ніколи не входять у форму.

</details>

---

## Частина 4 — TagForm і clean_name(): перша кастомна логіка
### Рівень: 🟡 Intermediate

### Ситуація

Користувач вводить назву тегу: `"  PYTHON  "`

Ми не хочемо зберігати це так. Ми хочемо: `"python"`

Як це зробити у Django Form? — Через метод `clean_<field>()`.

### Навіщо це потрібно?

Bez нормалізації у базі будуть теги `Python`, `python`, `PYTHON`, `  python  ` — всі різні!  
`clean_name()` гарантує що в базу потрапить тільки нормалізована форма.

### Схема: Validation Pipeline

При виклику `form.is_valid()` для кожного поля Django запускає pipeline:

```
raw_value (рядок з браузера)
    ↓
1. to_python(raw)    → Python тип (str / int / Notebook instance / ...)
    ↓
2. validate(value)   → перевірка required, max_length, choices
    ↓
3. run_validators()  → кастомні validators=[]
    ↓
4. clean_<field>()   → кастомна логіка (тільки якщо визначена)
    ↓
cleaned_data[field] = фінальне значення
    ↓
5. clean()           → крос-поле валідація (після всіх полів)
```

Ми додаємо логіку на кроці 4: `clean_name()` нормалізує після валідації Django.

In [ ]:
# ── to_python: перетворення сирого рядка → Python тип ────────────────────────
# Викликаємо to_python напряму щоб побачити що відбувається

form_temp = NoteForm(user=user)  # unbound — щоб отримати доступ до fields

# CharField: рядок → рядок (але stripped!)
title_field = form_temp.fields['title']
result = title_field.to_python('  Hello World  ')
print(f'CharField.to_python("  Hello World  ") → {result!r}')
print(f'Тип: {type(result).__name__}')
print()

# ModelChoiceField: рядок pk → ORM instance
notebook_field = form_temp.fields['notebook']
result2 = notebook_field.to_python(str(notebook.pk))
print(f'ModelChoiceField.to_python({notebook.pk!r}) → {result2!r}')
print(f'Тип: {type(result2).__name__}')

In [ ]:
# ── clean_name() у TagForm: нормалізація "  PYTHON  " → "python" ──────────────
# Дивись forms.py → class TagForm → def clean_name()

tag_form_ok = TagForm({'name': '  PYTHON  ', 'color': '#ff0000'})

print(f'is_valid(): {tag_form_ok.is_valid()}')
print(f'cleaned_data["name"]: {tag_form_ok.cleaned_data["name"]!r}')
print()
print('Що відбулось:')
print('  raw value:  "  PYTHON  "')
print('  to_python:  "  PYTHON  "   (CharField зберігає як є)')
print('  validate:   OK             (рядок не порожній)')
print('  clean_name: lower().strip() → "python"')

In [ ]:
# ── clean_name() з ValidationError: пробіли → порожньо після strip ────────────
# '   ' → required НЕ спрацьовує (рядок не порожній)
# → clean_name(): strip() → '' → raise ValidationError ← ось де!

tag_form_blank = TagForm({'name': '   ', 'color': '#ff0000'})
print(f'is_valid(): {tag_form_blank.is_valid()}')
print(f'errors:     {dict(tag_form_blank.errors)}')
print()

# Для порівняння: порожній рядок → required error від CharField.validate
tag_form_empty = TagForm({'name': '', 'color': '#ff0000'})
tag_form_empty.is_valid()
print(f'Порожній name errors: {dict(tag_form_empty.errors)}')
print('  ^ required помилка (від CharField.validate), не від clean_name!')

### Checkpoint 4

**Питання:** Що повернула б `clean_name()` якщо не зробити `return normalized`?

<details>
<summary>▶ Відповідь</summary>

Якщо метод `clean_<field>()` не повертає значення (або повертає `None`) → `cleaned_data['name'] = None`.

Forms очікує що `clean_<field>()` **завжди** повертає значення.  
Якщо `None` потрапить у сервіс і далі в базу — отримаємо `IntegrityError` або зламаний тег.

Правило: `clean_<field>()` **завжди** закінчується `return значення`.

</details>

---

## Частина 5 — NoteForm: ForeignKey, ManyToMany і user-фільтрація
### Рівень: 🟡 Intermediate

### Чому NoteForm складніша за NotebookForm?

| | NotebookForm | NoteForm |
|-|-|-|
| Поля | title, description, color, is_default | title, content, priority, **notebook**, **tags**, is_pinned |
| ForeignKey | ❌ | ✅ `notebook` → dropdown |
| ManyToMany | ❌ | ✅ `tags` → checkboxes |
| `__init__(user=)` | ❌ не потрібен | ✅ обов'язковий |

### Проблема без user-фільтрації

```python
# БЕЗ фільтрації — небезпечно!
self.fields['notebook'].queryset = Notebook.objects.all()
# Alice бачить у dropdown записники Bob'а і може вибрати їх!

# З фільтрацією — правильно:
self.fields['notebook'].queryset = Notebook.objects.filter(user=user)
# Alice бачить ТІЛЬКИ свої записники
```

In [ ]:
# ── user=None → порожній queryset ─────────────────────────────────────────────
# Якщо забути передати user — форма покаже порожній dropdown

form_no_user = NoteForm(user=None)

nb_qs = form_no_user.fields['notebook'].queryset
tags_qs = form_no_user.fields['tags'].queryset

print(f'user=None → notebook queryset: {nb_qs!r}')
print(f'user=None → tags queryset:     {tags_qs!r}')
print(f'  Кількість записників: {nb_qs.count()}')
print(f'  queryset is empty:    {not nb_qs.exists()}')

In [ ]:
# ── user=user → відфільтрований queryset ──────────────────────────────────────
# Тільки об'єкти цього користувача

form_with_user = NoteForm(user=user)

nb_qs = form_with_user.fields['notebook'].queryset
tags_qs = form_with_user.fields['tags'].queryset

print(f'user={user.username} → notebook queryset:')
print(f'  Записники: {list(nb_qs)}')
print(f'  SQL: {nb_qs.query}')
print()
print(f'user={user.username} → tags queryset:')
print(f'  Теги: {list(tags_qs)}')
print(f'  SQL: {tags_qs.query}')

### Що таке instance= ? Як форма знає що редагувати?

Коли користувач хоче **відредагувати** існуючу нотатку — форма має бути заповнена поточними значеннями.

Для цього передають `instance=note`:

```python
# Нова нотатка (CREATE):
form = NoteForm(user=request.user)

# Редагування (UPDATE):
form = NoteForm(instance=note, user=request.user)
#                ↑
# Django читає note.title, note.priority, note.notebook тощо
# і записує у form.initial
```

Після валідації `save()` робить UPDATE замість INSERT — але ми використовуємо `services.update_note()`.

In [ ]:
# ── instance= → форма заповнена поточними значеннями ─────────────────────────

edit_form = NoteForm(instance=note, user=user)

print(f'is_bound:  {edit_form.is_bound}')  # False — немає POST
print(f'initial:')
for k, v in edit_form.initial.items():
    print(f'  {k:15} = {v!r}')

In [ ]:
# ── POST + instance= → оновлення існуючого об'єкту ───────────────────────────

edit_data = {
    'title': 'Оновлена нотатка',
    'content': 'Новий вміст',
    'priority': '3',
    'notebook': str(notebook.pk),
    'tags': [str(tag_django.pk)],
    'is_pinned': True,
}

edit_bound = NoteForm(edit_data, instance=note, user=user)

print(f'is_valid(): {edit_bound.is_valid()}')
print(f'cleaned_data["title"]:    {edit_bound.cleaned_data["title"]!r}')
print(f'cleaned_data["priority"]: {edit_bound.cleaned_data["priority"]}  (тип: {type(edit_bound.cleaned_data["priority"]).__name__})')
print()
print('ВАЖЛИВО: не викликаємо edit_bound.save()!')
print('  → form_valid() передає cleaned_data у services.update_note()')
print('  → services.update_note() робить save() + транзакцію + side effects')

### Checkpoint 5

**Питання:** Чому `form_valid()` передає дані в `services.update_note()`, а не просто викликає `form.save()`?

<details>
<summary>▶ Відповідь</summary>

`form.save()` тільки зберігає дані в базу і нічого більше.

`services.update_note()` може:
- Зберегти у `transaction.atomic()` (захист від часткового збереження)
- Надіслати email повідомлення
- Оновити кеш
- Записати аудит-лог
- Викликати side effects на пов'язаних об'єктах

Сервісний шар — єдине місце де живе бізнес-логіка. Форма — лише валідація.

</details>

---

## Частина 6 — Widgets: як форма стає HTML
### Рівень: 🟡 Intermediate

### Звідки береться `<input>`?

Коли шаблон рендерить `{{ form.title }}`, Django:

```
forms.py
  ↓
NotebookForm.fields['title'] = CharField
  ↓
CharField.widget = TextInput(attrs={'class': 'form-control', ...})
  ↓
TextInput.render('title', value, attrs)
  ↓
{{ form.title }} у шаблоні
  ↓
<input type="text" name="title" class="form-control" ...>
```

### Widget Architecture

```
Python Field         Widget                 HTML
────────────         ──────                 ────
CharField        →   TextInput          →   <input type="text">
IntegerField     →   NumberInput        →   <input type="number">
BooleanField     →   CheckboxInput      →   <input type="checkbox">
ChoiceField      →   Select             →   <select><option>...
ModelChoiceField →   Select             →   <select> з queryset
M2MChoiceField   →   CheckboxSelectMultiple → окремий <input> для кожного
```

In [ ]:
# ── Рендеринг окремого поля ────────────────────────────────────────────────────
# str(field) → рендерить тільки <input>, без <label>

form_render = NoteForm(user=user)

title_html = str(form_render['title'])
print('form["title"] HTML:')
print(title_html)
print()
print('Атрибути прийшли з forms.py → class Meta → widgets:')
print('  class="form-control"  ← Bootstrap 5')
print('  placeholder="..."     ← з attrs')
print('  maxlength="200"       ← з моделі (max_length=200)')
print('  required              ← поле required=True')

In [ ]:
# ── Select widget для priority ─────────────────────────────────────────────────

priority_html = str(form_render['priority'])
print('form["priority"] HTML:')
print(priority_html)

In [ ]:
# ── Метадані BoundField ─────────────────────────────────────────────────────────
# BoundField = поле + значення + контекст форми

title_bound = form_render['title']

print(f'id_for_label:  {title_bound.id_for_label}')   # використовується у <label for="...">
print(f'label:         {title_bound.label}')           # текст для <label>
print(f'help_text:     {title_bound.help_text!r}')     # підказка під полем
print(f'field type:    {type(title_bound.field).__name__}')
print(f'widget type:   {type(title_bound.field.widget).__name__}')
print()
print('У шаблоні це використовується як:')
print('  <label for="{{ form.title.id_for_label }}">{{ form.title.label }}</label>')
print('  {{ form.title }}')
print('  <small>{{ form.title.help_text }}</small>')

In [ ]:
# ── Ітерація по CheckboxSelectMultiple (M:N теги) ─────────────────────────────
# У шаблоні: {% for checkbox in form.tags %} → кожен checkbox окремо

print('Ітерація по form["tags"] (кожен чекбокс окремо):')
for i, checkbox in enumerate(form_render['tags']):
    print(f'  [{i}] tag_id={checkbox.data["value"]}, label={checkbox.data["label"]}')
    print(f'       HTML: {str(checkbox)[:80]}...')
    print()

### Checkpoint 6

**Питання:** Як додати CSS клас `form-control-lg` до поля `title` без зміни HTML-шаблону?

<details>
<summary>▶ Відповідь</summary>

Через `attrs` у `widgets` у `class Meta`:

```python
class Meta:
    widgets = {
        'title': forms.TextInput(attrs={
            'class': 'form-control form-control-lg',
        })
    }
```

Або динамічно в `__init__`:

```python
def __init__(self, *args, **kwargs):
    super().__init__(*args, **kwargs)
    self.fields['title'].widget.attrs['class'] = 'form-control form-control-lg'
```

</details>

---

## Частина 7 — View: GET і POST цикл
### Рівень: 🟡 Intermediate

### GET і POST — два різних сценарії

```
GET /notes/new/    → показати порожню форму
POST /notes/new/   → прийняти дані → перевірити → зберегти або показати помилки
```

### Як це виглядає у FBV (Function-Based View)

```python
def note_create(request):
    if request.method == 'GET':
        form = NoteForm(user=request.user)         # порожня форма
        return render(request, 'note_form.html', {'form': form})

    if request.method == 'POST':
        form = NoteForm(request.POST, user=request.user)  # bound форма
        if form.is_valid():
            services.create_note(user=request.user, **form.cleaned_data)
            return redirect('hello_app:note_list')  # PRG pattern
        return render(request, 'note_form.html', {'form': form})  # помилки
```

`CreateView` робить **те саме автоматично**. Ми тільки перевизначаємо хуки.

### CBV хуки

| Хук | Коли викликається | Для чого |
|-----|-------------------|---------|
| `get_form_kwargs()` | GET і POST | Додає `user=` до аргументів форми |
| `form_valid()` | POST, форма валідна | Викликає services, робить redirect |
| `form_invalid()` | POST, форма невалідна | Рендерить форму з помилками (200) |
| `get_success_url()` | Після form_valid | Повертає URL для redirect |
| `get_context_data()` | GET і POST | Додає до контексту шаблону |

In [ ]:
# ── GET запит до NoteCreateView через RequestFactory ──────────────────────────
# RequestFactory: прямий виклик CBV без HTTP сервера і middleware

factory = RequestFactory()

request_get = factory.get('/notes/new/')
request_get.user = user  # ← ОБОВ'ЯЗКОВО! Немає middleware → user не встановлений

response_get = views.NoteCreateView.as_view()(request_get)

print(f'GET /notes/new/ → Status: {response_get.status_code}')
print(f'Response type: {type(response_get).__name__}')
print()
if hasattr(response_get, 'context_data') and response_get.context_data:
    ctx = response_get.context_data
    print(f'context_data keys: {list(ctx.keys())}')
    if 'form' in ctx:
        form_in_ctx = ctx['form']
        print(f'form type: {type(form_in_ctx).__name__}')
        print(f'form.is_bound: {form_in_ctx.is_bound}')  # False — GET → порожня форма

In [ ]:
# ── POST запит → NoteCreateView ───────────────────────────────────────────────
# ОБМЕЖЕННЯ RequestFactory: немає MessageMiddleware!
# form_valid() викликає messages.success() → потребує message storage
# Рішення: CookieStorage (не потребує сесій)

from django.contrib.messages.storage.cookie import CookieStorage

post_data = {
    'title': 'Нотатка з RequestFactory',
    'content': 'Тестування POST через factory',
    'priority': '1',
    'notebook': str(notebook.pk),
    'tags': [str(tag_python.pk)],
    'is_pinned': False,
}

post_request = factory.post('/notes/new/', data=post_data)
post_request.user = user
post_request._messages = CookieStorage(post_request)

response_post = views.NoteCreateView.as_view()(post_request)

print(f'POST /notes/new/ → Status: {response_post.status_code}')
if hasattr(response_post, 'url'):
    print(f'Redirect URL: {response_post.url}')
print()
print('302 → form_valid() спрацював → нотатку збережено!')
print('200 → форма невалідна, рендерить знову з помилками')

### Checkpoint 7

**Питання:** Навіщо `form_valid()` передає дані в `services` замість `form.save()`? Хіба `form.save()` не робить те саме?

<details>
<summary>▶ Відповідь</summary>

Дивись Checkpoint 5 — та сама відповідь застосовується і тут.

Додатково: `NoteCreateView.form_valid()` отримує `cleaned_data` з форми і викликає:  
`services.create_note(user=self.request.user, **form.cleaned_data)`

`user` встановлюється тут, на рівні View, — не з форми (захист від підробки).

</details>

---

## Частина 8 — Django Test Client: повний HTTP цикл
### Рівень: 🔴 Advanced

### Порівняння інструментів

| Інструмент | Що тестує | Middleware | URL routing | Context |
|------------|-----------|-----------|-----------|---------|
| `Form({data})` | Тільки валідацію | ❌ | ❌ | ❌ |
| `RequestFactory` | View логіку | ❌ | ❌ | частково |
| `Client()` | Повний HTTP цикл | ✅ | ✅ | ✅ |

**RequestFactory** — швидко, але без middleware (без сесій, без повного auth).

**Client** — повільніше, але реалістично — симулює реальний браузер:
- ✅ Сесії і cookies між запитами
- ✅ `response.context` — шаблонний контекст
- ✅ Відстежує redirect chain
- ✅ Перевіряє CSRF (якщо увімкнути)

In [ ]:
# Client використовує SERVER_NAME='testserver'
# → потрібно додати 'testserver' до ALLOWED_HOSTS

from django.conf import settings as django_settings
if 'testserver' not in django_settings.ALLOWED_HOSTS:
    django_settings.ALLOWED_HOSTS.append('testserver')

print('✅ testserver додано до ALLOWED_HOSTS')

In [ ]:
# ── GET запит через Client з force_login ──────────────────────────────────────

client = Client()
client.force_login(user)  # обходимо форму логіну

response = client.get('/notes/')

show_response(response, "GET /notes/ (після force_login)")

# Доступ до контексту шаблону
if response.context and 'notes' in response.context:
    notes_in_ctx = response.context['notes']
    print(f'  context["notes"]: {type(notes_in_ctx).__name__}')
    print(f'  Кількість нотаток: {len(notes_in_ctx)}')

In [ ]:
# ── Неавтентифікований запит → 302 redirect ───────────────────────────────────
# LoginRequiredMixin → /accounts/login/?next=/notes/

anon_client = Client()  # новий клієнт без force_login

response_unauth = anon_client.get('/notes/')
print(f'Анонімний GET /notes/ → Status: {response_unauth.status_code}')
if hasattr(response_unauth, 'url'):
    print(f'Redirect: {response_unauth.url}')

In [ ]:
# ── PRG pattern: POST → 302 → GET ─────────────────────────────────────────────
# Post/Redirect/Get: запобігає повторній відправці форми при F5

client2 = Client()
client2.force_login(user)

prg_data = {
    'title': 'Нотатка через Client',
    'content': 'PRG pattern demo',
    'priority': '1',
    'notebook': str(notebook.pk),
    'tags': [str(tag_python.pk)],
    'is_pinned': False,
}

post_response = client2.post('/notes/new/', data=prg_data)

print(f'POST /notes/new/ → Status: {post_response.status_code}')
if hasattr(post_response, 'url'):
    print(f'Redirect до: {post_response.url}')
print()
print('302 → form_valid() → get_success_url() → redirect до нової нотатки')

In [ ]:
# ── Невалідний POST → 200 з помилками у context ───────────────────────────────
# Якщо форма невалідна → View рендерить форму знову (200, не 302)

invalid_post = {
    'title': '',      # ← порожній! required
    'content': '',
    'priority': '1',
    'notebook': '',
}

invalid_response = client2.post('/notes/new/', data=invalid_post)

print(f'Невалідний POST → Status: {invalid_response.status_code}')
print('200 (не 302!) → форма рендериться знову з помилками')
print()
if invalid_response.context and 'form' in invalid_response.context:
    ctx_form = invalid_response.context['form']
    print(f'form.errors: {dict(ctx_form.errors)}')
    print(f'form.is_bound: {ctx_form.is_bound}')

In [ ]:
# ── reverse() і Client: не хардкодимо URL ────────────────────────────────────

url_list   = reverse('hello_app:note_list')
url_create = reverse('hello_app:note_create')
url_detail = reverse('hello_app:note_detail', kwargs={'pk': note.pk})
url_nb     = reverse('hello_app:notebook_list')

print('URL reverse():')
print(f'  note_list:   {url_list}')
print(f'  note_create: {url_create}')
print(f'  note_detail: {url_detail}')
print(f'  nb_list:     {url_nb}')
print()

client3 = Client()
client3.force_login(user)

r1 = client3.get(reverse('hello_app:note_list'))
r2 = client3.get(reverse('hello_app:note_detail', kwargs={'pk': note.pk}))
r3 = client3.get(reverse('hello_app:notebook_list'))

print('Client з reverse():')
print(f'  GET note_list:     {r1.status_code}')
print(f'  GET note_detail:   {r2.status_code}')
print(f'  GET notebook_list: {r3.status_code}')

---
## Частина 9 — CSRF у тестах
### Рівень: 🔴 Advanced

**CSRF (Cross-Site Request Forgery)** — механізм захисту від підробки запитів.

Django вбудовує у форму прихований токен:

    <input type="hidden" name="csrfmiddlewaretoken" value="abc123...">

При POST — сервер перевіряє що токен є і правильний.

У тестах:
- `Client()` за замовчуванням **ігнорує CSRF** — зручно для тестування логіки
- `Client(enforce_csrf_checks=True)` — вмикає реальну перевірку

In [ ]:
# ── За замовчуванням Client ігнорує CSRF ──────────────────────────────────────
# enforce_csrf_checks=False (дефолт) → CSRF не перевіряється

client_no_csrf = Client()  # enforce_csrf_checks=False за замовчуванням
client_no_csrf.force_login(user)

r = client_no_csrf.post('/notes/new/', data={
    'title': 'CSRF test',
    'content': '',
    'priority': '1',
    'notebook': '',
})
print(f'Без CSRF token → Status: {r.status_code}')
print('Не 403 — CSRF не перевіряється у тестах за замовчуванням')

In [ ]:
# ── enforce_csrf_checks=True → 403 без токену ─────────────────────────────────

client_csrf = Client(enforce_csrf_checks=True)
client_csrf.force_login(user)

r_403 = client_csrf.post('/notes/new/', data={
    'title': 'CSRF test',
    'content': '',
    'priority': '1',
    'notebook': '',
})
print(f'З enforce_csrf_checks=True, без токену → Status: {r_403.status_code}')
print('403 Forbidden — CSRF middleware заблокував запит!')

In [ ]:
# ── RequestFactory: CSRF не перевіряється взагалі ─────────────────────────────
# RequestFactory не проганяє middleware → CsrfViewMiddleware не запускається
# Щоб реально тестувати CSRF → використовуй Client(enforce_csrf_checks=True)

print('RequestFactory + enforce_csrf_checks:')
print('  → middleware не в stack → CSRF не перевіряється')
print()
print('Для реальної перевірки CSRF → Client(enforce_csrf_checks=True)')
print()
print('Коли вмикати CSRF тести?')
print('  ✅ Якщо хочеш перевірити що форма має CSRF захист')
print('  ❌ Не потрібно в кожному тесті — сповільнює і ускладнює')

### Checkpoint 9

**Питання:** Чому `RequestFactory` швидший але менш реалістичний ніж `Client`?

<details>
<summary>▶ Відповідь</summary>

`RequestFactory` — прямий виклик функції View, без middleware stack:
- Не запускає SessionMiddleware → немає сесій
- Не запускає AuthenticationMiddleware → `request.user` треба встановити вручну
- Не запускає MessageMiddleware → треба додавати `CookieStorage` вручну
- Не робить URL routing → немає `request.resolver_match`

`Client` симулює повний HTTP цикл через Django WSGI handler — так само як реальний браузер.

Правило: використовуй `RequestFactory` для швидких тестів View-логіки,  
а `Client` для інтеграційних тестів (auth, redirect chains, template context).

</details>

---

## Частина 10 — Підсумок

### Testing Matrix: що і як тестувати

```
┌──────────────────────┬────────────────────────────────────────────────┐
│ Що перевіряємо       │ Як тестуємо                                    │
├──────────────────────┼────────────────────────────────────────────────┤
│ Form validation      │ Прямий виклик Form({data})                     │
│ cleaned_data types   │ form.cleaned_data після is_valid()             │
│ clean_<field>()      │ Form з конкретними даними                      │
│ ValidationError      │ form.errors після is_valid()                   │
├──────────────────────┼────────────────────────────────────────────────┤
│ View логіка (швидко) │ RequestFactory + request.user                  │
│ GET params           │ factory.get(path, {'q': 'value'})              │
│ AnonymousUser        │ request.user = AnonymousUser()                 │
├──────────────────────┼────────────────────────────────────────────────┤
│ Full HTTP цикл       │ Client() + force_login()                       │
│ login_required       │ Client без force_login → 302                   │
│ PRG pattern          │ client.post() → 302 → client.get()             │
│ Template context     │ response.context['key']                        │
│ Template names       │ response.templates                             │
│ CSRF                 │ Client(enforce_csrf_checks=True)               │
├──────────────────────┼────────────────────────────────────────────────┤
│ Реальний браузер     │ Selenium / Playwright (окрема тема)            │
└──────────────────────┴────────────────────────────────────────────────┘
```

In [ ]:
# ── Фінальна перевірка: що створили під час лабораторії ──────────────────────
print("Об'єкти створені під час лабораторії:")
print()

notes = Note.objects.filter(user=user)
notebooks = Notebook.objects.filter(user=user)
tags = Tag.objects.filter(user=user)

print(f'Notes ({notes.count()}):')
for n in notes:
    print(f'  [{n.pk}] {n.title} (priority={n.priority})')

print(f'\nNotebooks ({notebooks.count()}):')
for nb in notebooks:
    print(f'  [{nb.pk}] {nb.title}')

print(f'\nTags ({tags.count()}):')
for t in tags:
    print(f'  [{t.pk}] {t.name} ({t.color})')

### Подорож нотатки — фінальна схема

```
Користувач відкриває /notes/new/  →  GET запит
                                          ↓
                             NoteCreateView.get()
                                          ↓
                         form = NoteForm(user=request.user)
                                          ↓
                     template: {{ form.title }}, {{ form.tags }}...
                                          ↓
                             HTML → браузер показує форму

──────────────────────────────────────────────────────────

Користувач заповнює і натискає Зберегти  →  POST запит
                                          ↓
                            NoteCreateView.post()
                                          ↓
                form = NoteForm(request.POST, user=request.user)
                                          ↓
                            form.is_valid()?
                           ╱              ╲
                     False                True
                       ↓                    ↓
              render (200)          services.create_note()
              з помилками                   ↓
                                      transaction.atomic()
                                           ↓
                                       Database
                                           ↓
                               redirect 302 → /notes/{pk}/
                                           ↓
                                  Користувач бачить нотатку
```

---

📚 **Документація:**
- [DJANGO_FORMS.md](../../DJANGO_FORMS.md) — Trust Boundary, Validation Pipeline
- [README.md](../README.md) — CBV: as\_view(), dispatch(), Generic Views
- [DJANGO_SERVICES.md](../../lesson_Django_Network_Architecture/DJANGO_SERVICES.md) — Services шар

**Наступний крок:** `pytest` + `django.test.TestCase` — автоматизовані тести замість ручного запуску у notebook.